In [1]:
# @title Cell 1: Install All Dependencies
!pip install torch flask pyngrok sentence-transformers python-docx pdfplumber nltk --quiet
!pip install requests beautifulsoup4 reportlab --quiet
!pip install PyPDF2 pytesseract pillow opencv-python --quiet

# Install Tesseract for OCR
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y poppler-utils

print("✅ All packages installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.2 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease

In [2]:
# @title Cell 2: Import All Libraries
from flask import Flask, render_template_string, request, jsonify, send_file
from pyngrok import ngrok
from sentence_transformers import SentenceTransformer, util
from docx import Document
import pdfplumber
import PyPDF2
import os
import tempfile
import re
import nltk
import time
import torch
import requests
import json
import io
import hashlib
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [3]:
# @title Cell 3: Configure API Keys and Model

# Your ngrok authtoken
ngrok.set_auth_token("3AvojTQh3zjit4QIJYB64020Mgs_53Bg78hTn2xV1preKPugU")

# Your Serper.dev API key
SERPER_API_KEY = "8eff6c07fe697d3541a8e46036581cc1ff6842dc"
YOUR_EMAIL = "ashishnayakofficial05@gmail.com"

# Load the sentence transformer model for similarity (all-mpnet-base-v2)
print("Loading sentence transformer model (all-mpnet-base-v2)...")
model = SentenceTransformer('all-mpnet-base-v2')
print("✅ Model loaded successfully!")

print(f"✅ Serper API Key configured: {SERPER_API_KEY[:10]}...")
print(f"✅ Email configured: {YOUR_EMAIL}")

Loading sentence transformer model (all-mpnet-base-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded successfully!
✅ Serper API Key configured: 8eff6c07fe...
✅ Email configured: ashishnayakofficial05@gmail.com


In [4]:
# @title Cell 4: Web Search API Class (Serper.dev + OpenAlex) - FIXED CONFIDENCE

class PlagiarismSearchAPI:
    """
    Combined API integration for Serper.dev (web search) and OpenAlex (academic papers)
    """

    def __init__(self, serper_api_key: str, openalex_email: str):
        self.serper_api_key = serper_api_key
        self.serper_base_url = "https://google.serper.dev/search"
        self.openalex_base_url = "https://api.openalex.org"
        self.openalex_email = openalex_email

        self.serper_headers = {
            'X-API-KEY': self.serper_api_key,
            'Content-Type': 'application/json'
        }

        self.last_request_time = 0
        self.min_request_interval = 1.0  # 1 second to avoid rate limits

    def _rate_limit(self):
        now = time.time()
        if now - self.last_request_time < self.min_request_interval:
            time.sleep(self.min_request_interval - (now - self.last_request_time))
        self.last_request_time = time.time()

    def _normalize_confidence(self, confidence):
        """Ensure confidence is always between 0 and 1"""
        if confidence is None:
            return 0.0
        # If it's already a percentage (e.g., 62.1), convert to fraction
        if confidence > 1:
            confidence = confidence / 100.0
        return max(0.0, min(1.0, confidence))

    def _calculate_relevance(self, query: str, text: str) -> float:
        """Calculate relevance score between query and text (returns 0-1)"""
        if not text:
            return 0.0
        query_words = set(query.lower().split())
        text_words = set(text.lower().split())
        if not query_words:
            return 0.0
        common_words = query_words.intersection(text_words)
        phrase_match = 1.0 if query.lower() in text.lower() else 0.0
        overlap_score = len(common_words) / len(query_words)
        final_score = (overlap_score * 0.7) + (phrase_match * 0.3)
        return min(final_score, 1.0)

    def search_web(self, query: str, num_results: int = 5):
        """Search the web using Serper.dev"""
        self._rate_limit()
        try:
            clean_query = ' '.join(query.split())[:200]
            payload = json.dumps({
                "q": clean_query,
                "num": min(num_results, 10),
                "gl": "us",
                "hl": "en"
            })
            response = requests.post(
                self.serper_base_url,
                headers=self.serper_headers,
                data=payload,
                timeout=15
            )
            response.raise_for_status()
            data = response.json()
            results = []
            for item in data.get('organic', [])[:num_results]:
                snippet = item.get('snippet', '')
                title = item.get('title', '')
                confidence = self._calculate_relevance(query, snippet)
                title_conf = self._calculate_relevance(query, title)
                confidence = max(confidence, title_conf * 0.8)
                # Normalize confidence (already in 0-1, but ensure)
                confidence = self._normalize_confidence(confidence)
                results.append({
                    'title': title,
                    'url': item.get('link', ''),
                    'snippet': snippet,
                    'source_type': 'web',
                    'api_source': 'serper',
                    'confidence': round(confidence, 2),
                })
            results.sort(key=lambda x: x['confidence'], reverse=True)
            return results
        except Exception as e:
            print(f"Serper API error: {e}")
            return []

    def search_academic(self, query: str, num_results: int = 5):
        """Search academic papers using OpenAlex"""
        self._rate_limit()
        try:
            clean_query = ' '.join(query.split())[:200]
            url = f"{self.openalex_base_url}/works"
            params = {
                'search': clean_query,
                'per-page': min(num_results, 25),
                'sort': 'relevance_score:desc',
                'mailto': self.openalex_email
            }
            response = requests.get(url, params=params, timeout=15)
            if response.status_code == 500:
                print(f"OpenAlex server error for query: {clean_query[:50]}...")
                return []
            response.raise_for_status()
            data = response.json()
            results = []
            for work in data.get('results', [])[:num_results]:
                doi = work.get('doi', '')
                doi_url = f"https://doi.org/{doi}" if doi else ''
                best_url = doi_url
                if not best_url and work.get('primary_location', {}).get('landing_page_url'):
                    best_url = work['primary_location']['landing_page_url']
                authors = []
                for authorship in work.get('authorships', [])[:3]:
                    author = authorship.get('author', {})
                    if author.get('display_name'):
                        authors.append(author['display_name'])
                title = work.get('title', '')
                # OpenAlex sometimes gives relevance_score as percentage (0-100)
                raw_confidence = work.get('relevance_score', 0.5)
                confidence = self._normalize_confidence(raw_confidence)
                if confidence < 0.3 and title:
                    confidence = self._calculate_relevance(query, title)
                results.append({
                    'title': title,
                    'url': best_url,
                    'doi': doi,
                    'authors': authors,
                    'publication_year': work.get('publication_year'),
                    'source_type': 'academic',
                    'api_source': 'openalex',
                    'confidence': round(confidence, 2)
                })
            results.sort(key=lambda x: x['confidence'], reverse=True)
            return results
        except Exception as e:
            print(f"OpenAlex API error: {e}")
            return []

    def search_all(self, query: str, num_results: int = 3):
        """Search both web and academic sources"""
        web_results = self.search_web(query, num_results)
        academic_results = self.search_academic(query, num_results)
        all_results = web_results + academic_results
        all_results.sort(key=lambda x: x.get('confidence', 0), reverse=True)
        return {
            'web_results': web_results,
            'academic_results': academic_results,
            'all_results': all_results,
            'best_match': all_results[0] if all_results else None
        }

# Initialize the search API
search_api = PlagiarismSearchAPI(SERPER_API_KEY, YOUR_EMAIL)
print("✅ Web Search API initialized (Serper.dev + OpenAlex) with confidence normalization")

✅ Web Search API initialized (Serper.dev + OpenAlex) with confidence normalization


In [5]:
# @title Cell 5: Document Analyzer for Web Search - FIXED

class DocumentAnalyzer:
    """
    Analyzes document chunks using the search APIs
    """

    def __init__(self, serper_api_key: str, openalex_email: str):
        self.search_api = PlagiarismSearchAPI(serper_api_key, openalex_email)
        print("✅ DocumentAnalyzer initialized")

    def extract_text_from_pdf(self, pdf_file):
        """Extract text from uploaded PDF file"""
        text_blocks = []
        try:
            pdf_file.seek(0)
            with pdfplumber.open(io.BytesIO(pdf_file.read())) as pdf:
                for page_num, page in enumerate(pdf.pages):
                    text = page.extract_text()
                    if text:
                        text = ' '.join(text.split())
                        text_blocks.append({'text': text, 'page': page_num})
            print(f"✅ Extracted {len(text_blocks)} text blocks from PDF")
            return text_blocks
        except Exception as e:
            print(f"Error extracting PDF: {e}")
            return []

    def chunk_text(self, text_blocks, chunk_size=50):
        """Split text into chunks for analysis"""
        full_text = ' '.join([block['text'] for block in text_blocks])
        words = full_text.split()
        chunks = []
        overlap = 10
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if len(chunk.split()) > 10:
                chunks.append(chunk)
        print(f"✅ Created {len(chunks)} text chunks for analysis")
        return chunks

    def analyze_chunk(self, chunk_text: str, chunk_id: int = 0):
        """Analyze a single text chunk - ensures confidence is normalized"""
        if len(chunk_text.split()) < 10:
            return {
                'chunk_id': chunk_id,
                'text': chunk_text,
                'has_matches': False,
                'best_match': None,
                'source_type': 'original',
                'highlight_color': None,
                'confidence': 0
            }
        try:
            search_results = self.search_api.search_all(chunk_text, num_results=3)
            source_type = 'original'
            highlight_color = None
            best_match = None
            if search_results and search_results.get('best_match'):
                best_match = search_results['best_match']
                # Force confidence to be between 0 and 1
                if best_match and 'confidence' in best_match:
                    conf = best_match['confidence']
                    if conf > 1:
                        conf = conf / 100.0
                    conf = max(0.0, min(1.0, conf))
                    best_match['confidence'] = round(conf, 2)
                if best_match and best_match.get('confidence', 0) > 0.25:
                    if best_match.get('source_type') == 'web':
                        source_type = 'web'
                        highlight_color = 'web-text'
                    elif best_match.get('source_type') == 'academic':
                        source_type = 'academic'
                        highlight_color = 'paper-text'
            return {
                'chunk_id': chunk_id,
                'text': chunk_text,
                'has_matches': bool(search_results and (search_results.get('web_results') or search_results.get('academic_results'))),
                'best_match': best_match,
                'source_type': source_type,
                'highlight_color': highlight_color,
                'confidence': best_match.get('confidence', 0) if best_match else 0
            }
        except Exception as e:
            print(f"Error analyzing chunk {chunk_id}: {e}")
            return {
                'chunk_id': chunk_id,
                'text': chunk_text,
                'has_matches': False,
                'best_match': None,
                'source_type': 'original',
                'highlight_color': None,
                'confidence': 0
            }

    def analyze_document(self, pdf_file):
        """Analyze entire document"""
        try:
            text_blocks = self.extract_text_from_pdf(pdf_file)
            if not text_blocks:
                print("❌ No text extracted from PDF")
                return None
            chunks = self.chunk_text(text_blocks, chunk_size=40)
            chunk_results = []
            web_count = 0
            academic_count = 0
            all_matched_sources = []
            print(f"📄 Analyzing {len(chunks)} chunks...")
            for i, chunk in enumerate(chunks):
                if i % 5 == 0:
                    print(f"   Processing chunk {i+1}/{len(chunks)}...")
                result = self.analyze_chunk(chunk, i)
                chunk_results.append(result)
                if result['source_type'] == 'web':
                    web_count += 1
                elif result['source_type'] == 'academic':
                    academic_count += 1
                if result.get('best_match'):
                    all_matched_sources.append(result['best_match'])
            total_chunks = len(chunks)
            # Remove duplicate sources
            unique_sources = []
            seen_urls = set()
            for source in all_matched_sources:
                url = source.get('url', source.get('doi', ''))
                if url and url not in seen_urls:
                    seen_urls.add(url)
                    unique_sources.append(source)
            print(f"✅ Analysis complete: {web_count} web matches, {academic_count} academic matches")
            return {
                'total_chunks': total_chunks,
                'web_percentage': round((web_count / total_chunks) * 100, 1) if total_chunks > 0 else 0,
                'academic_percentage': round((academic_count / total_chunks) * 100, 1) if total_chunks > 0 else 0,
                'overall_similarity': round(((web_count + academic_count) / total_chunks) * 100, 1) if total_chunks > 0 else 0,
                'chunk_results': chunk_results,
                'all_matched_sources': unique_sources[:20]
            }
        except Exception as e:
            print(f"Error analyzing document: {e}")
            return None

    def generate_highlighted_html(self, analysis_result):
        """Generate HTML with color-coded highlighting"""
        html_parts = []
        for chunk in analysis_result['chunk_results']:
            color_class = chunk.get('highlight_color', '')
            tooltip = ""
            if chunk.get('best_match'):
                best = chunk['best_match']
                if best.get('source_type') == 'web':
                    tooltip = f"🌐 Web Source: {best.get('title', '')}\nURL: {best.get('url', '')}"
                else:
                    authors = ', '.join(best.get('authors', ['Unknown'])[:2])
                    tooltip = f"📚 Academic: {best.get('title', '')}\nAuthors: {authors}"
            if color_class and tooltip:
                html_parts.append(f'<span class="{color_class}" title="{tooltip}">{chunk["text"]}</span>')
            elif color_class:
                html_parts.append(f'<span class="{color_class}">{chunk["text"]}</span>')
            else:
                html_parts.append(f'<span>{chunk["text"]}</span>')
        return '<br><br>'.join(html_parts)

# Re-initialize the analyzer
analyzer = DocumentAnalyzer(SERPER_API_KEY, YOUR_EMAIL)
print("\n✅ Web Search Document Analyzer ready with confidence normalization")

✅ DocumentAnalyzer initialized

✅ Web Search Document Analyzer ready with confidence normalization


In [6]:
# @title Cell 6: Helper Functions for Similarity Detection

def read_file(file):
    """Read text from uploaded file (TXT, PDF, DOCX)"""
    if not file or not file.filename:
        return ""

    ext = os.path.splitext(file.filename)[1].lower()
    with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
        tmp.write(file.read())
        path = tmp.name

    text = ""
    try:
        if ext == ".txt":
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
        elif ext == ".pdf":
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    text += page.extract_text() or ""
        elif ext == ".docx":
            doc = Document(path)
            text = "\n".join([p.text for p in doc.paragraphs])
    except Exception as e:
        print(f"Error reading file {file.filename}: {e}")
        return ""
    finally:
        os.remove(path)
    return text.strip()

def highlight_similar_text(source_text, compare_text, similarity_threshold=0.7):
    """
    Highlight similar text segments between source and compare text
    """
    try:
        # Split texts into sentences
        source_sentences = nltk.sent_tokenize(source_text)
        compare_sentences = nltk.sent_tokenize(compare_text)

        if not source_sentences or not compare_sentences:
            return source_text, compare_text, []

        # Encode sentences
        source_embs = model.encode(source_sentences, convert_to_tensor=True)
        compare_embs = model.encode(compare_sentences, convert_to_tensor=True)

        # Find similar sentences
        similar_segments = []
        highlighted_source = source_text
        highlighted_compare = compare_text

        for i, source_sent in enumerate(source_sentences):
            similarities = util.cos_sim(source_embs[i], compare_embs)
            max_similarity, max_idx = torch.max(similarities, dim=1)
            max_similarity = max_similarity.item()
            max_idx = max_idx.item()

            if max_similarity >= similarity_threshold:
                compare_sent = compare_sentences[max_idx]

                similar_segments.append({
                    'source_sentence': source_sent,
                    'compare_sentence': compare_sent,
                    'similarity': max_similarity
                })

                # Highlight in source text
                highlighted_source = highlighted_source.replace(
                    source_sent,
                    f'<span class="plagiarism-highlight">{source_sent}</span>'
                )

                # Highlight in compare text
                highlighted_compare = highlighted_compare.replace(
                    compare_sent,
                    f'<span class="plagiarism-highlight">{compare_sent}</span>'
                )

        return highlighted_source, highlighted_compare, similar_segments

    except Exception as e:
        print(f"Error in highlighting: {e}")
        return source_text, compare_text, []

print("✅ Similarity detection helpers loaded")

✅ Similarity detection helpers loaded


In [7]:
# @title Cell 7: Flask Application with Enhanced PDF Report (Confidence Fixed)

app = Flask(__name__)

# Store the last analysis results for report generation
last_web_results = None
last_web_sources = None
last_filename = None
last_upload_time = None

# HTML Template with two separate features
HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Academic Integrity System</title>
    <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css" rel="stylesheet">
    <style>
        /* (same CSS as before – omitted for brevity, but keep your existing CSS) */
        @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Inter:wght@300;400;600;700&display=swap');

        :root {
            --gold: #FFD700;
            --dark-gold: #B8860B;
            --black: #0A0A0A;
            --dark-gray: #1A1A1A;
            --medium-gray: #2A2A2A;
            --light-gray: #3A3A3A;
            --neon-gold: #FFEE58;
            --plagiarism-red: #ff4444;
            --plagiarism-orange: #ff8800;
            --web-blue: #3182ce;
            --academic-green: #38a169;
        }

        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: 'Inter', sans-serif;
            background: linear-gradient(135deg, var(--black) 0%, #1a1a1a 100%);
            min-height: 100vh;
            padding: 20px;
            color: white;
        }

        .container {
            max-width: 1400px;
            margin: 0 auto;
            background: rgba(26, 26, 26, 0.95);
            border-radius: 20px;
            box-shadow: 0 25px 50px rgba(0,0,0,0.5);
            overflow: hidden;
            border: 1px solid rgba(255, 215, 0, 0.2);
        }

        .header {
            background: linear-gradient(135deg, var(--black) 0%, var(--dark-gray) 100%);
            color: var(--gold);
            padding: 40px 30px;
            text-align: center;
            border-bottom: 2px solid var(--gold);
        }

        .header h1 {
            font-family: 'Playfair Display', serif;
            font-size: 3em;
            margin-bottom: 15px;
            background: linear-gradient(45deg, var(--gold), var(--neon-gold));
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
        }

        .header p {
            color: #ccc;
            font-size: 1.1em;
        }

        .feature-tabs {
            display: flex;
            gap: 10px;
            margin: 30px 30px 20px 30px;
            background: var(--dark-gray);
            padding: 10px;
            border-radius: 15px;
        }

        .feature-tab {
            flex: 1;
            padding: 15px 20px;
            border: 2px solid var(--medium-gray);
            background: transparent;
            color: var(--gold);
            border-radius: 10px;
            font-size: 16px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            text-align: center;
        }

        .feature-tab.active {
            background: linear-gradient(135deg, var(--gold), var(--dark-gold));
            color: var(--black);
            border-color: var(--gold);
        }

        .feature-tab i {
            margin-right: 8px;
        }

        .feature-content {
            padding: 0 30px 30px 30px;
        }

        .feature-panel {
            display: none;
        }

        .feature-panel.active {
            display: block;
        }

        /* Web Search Feature Styles */
        .web-search-header {
            background: linear-gradient(135deg, #1a1a1a, #2a2a2a);
            border: 2px solid var(--web-blue);
            border-radius: 15px;
            padding: 25px;
            margin-bottom: 30px;
        }

        .web-search-header h2 {
            color: var(--web-blue);
            margin-bottom: 15px;
            font-size: 1.8em;
        }

        .upload-area {
            background: rgba(42, 42, 42, 0.8);
            border: 3px dashed var(--web-blue);
            border-radius: 15px;
            padding: 50px 20px;
            text-align: center;
            margin: 20px 0;
            cursor: pointer;
            transition: all 0.3s ease;
        }

        .upload-area:hover {
            background: rgba(49, 130, 206, 0.1);
            border-color: var(--neon-gold);
        }

        .upload-icon {
            font-size: 48px;
            color: var(--web-blue);
            margin-bottom: 15px;
        }

        .stats-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 20px;
            margin: 30px 0;
        }

        .stat-card {
            background: var(--dark-gray);
            padding: 20px;
            border-radius: 10px;
            text-align: center;
            border: 1px solid var(--medium-gray);
        }

        .stat-value {
            font-size: 2.5em;
            font-weight: bold;
            color: var(--gold);
        }

        .legend {
            display: flex;
            gap: 20px;
            margin: 20px 0;
            padding: 15px;
            background: var(--dark-gray);
            border-radius: 10px;
            justify-content: center;
        }

        .legend-item {
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .color-box {
            width: 20px;
            height: 20px;
            border-radius: 4px;
        }

        .web-text {
            background-color: rgba(49, 130, 206, 0.3);
            border-bottom: 2px solid var(--web-blue);
            padding: 2px 4px;
            border-radius: 3px;
            cursor: help;
        }

        .paper-text {
            background-color: rgba(56, 161, 105, 0.3);
            border-bottom: 2px solid var(--academic-green);
            padding: 2px 4px;
            border-radius: 3px;
            cursor: help;
        }

        .highlight-box {
            background: var(--dark-gray);
            border: 1px solid var(--medium-gray);
            border-radius: 10px;
            padding: 20px;
            max-height: 400px;
            overflow-y: auto;
            font-family: 'Courier New', monospace;
            line-height: 1.6;
            margin: 20px 0;
        }

        .sources-list {
            list-style: none;
            margin-top: 20px;
        }

        .source-item {
            background: var(--dark-gray);
            border: 1px solid var(--medium-gray);
            border-radius: 10px;
            padding: 15px;
            margin-bottom: 10px;
            transition: transform 0.2s;
        }

        .source-item:hover {
            transform: translateX(5px);
            border-color: var(--gold);
        }

        .source-title {
            font-weight: bold;
            color: var(--gold);
            margin-bottom: 5px;
        }

        .source-url {
            color: var(--web-blue);
            text-decoration: none;
            font-size: 0.9em;
            word-break: break-all;
        }

        .btn {
            background: linear-gradient(135deg, var(--gold), var(--dark-gold));
            color: var(--black);
            border: none;
            padding: 12px 30px;
            border-radius: 25px;
            font-size: 1em;
            font-weight: bold;
            cursor: pointer;
            transition: transform 0.2s;
            margin-top: 20px;
        }

        .btn:hover {
            transform: translateY(-2px);
            box-shadow: 0 5px 15px rgba(255, 215, 0, 0.3);
        }

        .btn-web {
            background: linear-gradient(135deg, var(--web-blue), #1e4a6b);
            color: white;
        }

        /* Similarity Detection Feature Styles */
        .similarity-header {
            background: linear-gradient(135deg, #1a1a1a, #2a2a2a);
            border: 2px solid var(--gold);
            border-radius: 15px;
            padding: 25px;
            margin-bottom: 30px;
        }

        .similarity-header h2 {
            color: var(--gold);
            margin-bottom: 15px;
            font-size: 1.8em;
        }

        .mode-tabs {
            display: flex;
            gap: 10px;
            margin-bottom: 30px;
            background: var(--dark-gray);
            padding: 10px;
            border-radius: 15px;
        }

        .mode-tab {
            flex: 1;
            padding: 12px 15px;
            border: 2px solid var(--medium-gray);
            background: transparent;
            color: var(--gold);
            border-radius: 8px;
            font-size: 14px;
            font-weight: 600;
            cursor: pointer;
            transition: all 0.3s ease;
            text-align: center;
        }

        .mode-tab.active {
            background: linear-gradient(135deg, var(--gold), var(--dark-gold));
            color: var(--black);
            border-color: var(--gold);
        }

        .tab-content {
            display: none;
        }

        .tab-content.active {
            display: block;
        }

        .input-section {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 30px;
            margin-bottom: 30px;
        }

        .text-input {
            border: 2px solid var(--medium-gray);
            border-radius: 15px;
            padding: 25px;
            background: rgba(42, 42, 42, 0.8);
        }

        .text-input h3 {
            margin-bottom: 20px;
            color: var(--gold);
            font-family: 'Playfair Display', serif;
            font-size: 1.3em;
        }

        textarea, input[type="file"] {
            width: 100%;
            padding: 15px;
            border: 1px solid var(--light-gray);
            border-radius: 10px;
            background: var(--dark-gray);
            color: white;
            font-family: 'Inter', sans-serif;
        }

        textarea {
            height: 200px;
            resize: vertical;
        }

        .file-list {
            max-height: 200px;
            overflow-y: auto;
            margin-top: 15px;
        }

        .file-item {
            background: var(--dark-gray);
            padding: 10px;
            margin: 5px 0;
            border-radius: 5px;
            border-left: 4px solid var(--gold);
            display: flex;
            justify-content: space-between;
            align-items: center;
        }

        .remove-file {
            background: #dc3545;
            color: white;
            border: none;
            border-radius: 50%;
            width: 25px;
            height: 25px;
            cursor: pointer;
        }

        .plagiarism-highlight {
            background: linear-gradient(135deg, var(--plagiarism-red), var(--plagiarism-orange));
            color: white;
            padding: 2px 4px;
            border-radius: 3px;
            font-weight: bold;
            animation: pulse-highlight 2s infinite;
        }

        @keyframes pulse-highlight {
            0%, 100% { opacity: 1; }
            50% { opacity: 0.8; }
        }

        .text-comparison {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
            margin-top: 20px;
        }

        .text-column {
            background: var(--medium-gray);
            padding: 15px;
            border-radius: 8px;
            border: 1px solid var(--light-gray);
        }

        .text-column h6 {
            color: var(--neon-gold);
            margin-bottom: 10px;
            border-bottom: 1px solid var(--light-gray);
            padding-bottom: 5px;
        }

        .highlighted-text {
            max-height: 300px;
            overflow-y: auto;
            line-height: 1.6;
            font-size: 14px;
        }

        .comparison-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(350px, 1fr));
            gap: 20px;
            margin-top: 20px;
        }

        .comparison-card {
            background: var(--dark-gray);
            padding: 20px;
            border-radius: 15px;
            border: 1px solid var(--medium-gray);
        }

        .similarity-badge {
            padding: 10px;
            border-radius: 8px;
            text-align: center;
            font-weight: bold;
            margin-bottom: 15px;
        }

        .badge-high {
            background: #dc3545;
            color: white;
        }

        .badge-medium {
            background: #ffc107;
            color: black;
        }

        .badge-low {
            background: #28a745;
            color: white;
        }

        .expand-btn {
            background: var(--gold);
            color: var(--black);
            border: none;
            border-radius: 5px;
            padding: 8px 15px;
            cursor: pointer;
            margin-top: 10px;
            font-weight: bold;
        }

        .hidden {
            display: none;
        }

        .progress {
            height: 25px;
            background: var(--medium-gray);
            border-radius: 10px;
            overflow: hidden;
            margin: 20px 0;
        }

        .progress-bar {
            height: 100%;
            background: linear-gradient(90deg, var(--gold), var(--neon-gold));
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: bold;
            color: var(--black);
            transition: width 0.3s ease;
        }

        .loading {
            text-align: center;
            padding: 40px;
        }

        .spinner {
            border: 4px solid #f3f3f3;
            border-top: 4px solid var(--web-blue);
            border-radius: 50%;
            width: 50px;
            height: 50px;
            animation: spin 1s linear infinite;
            margin: 0 auto 20px;
        }

        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }

        @media (max-width: 768px) {
            .input-section,
            .text-comparison {
                grid-template-columns: 1fr;
            }
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1><i class="fas fa-graduation-cap"></i> Academic Integrity System</h1>
            <p>Choose your detection method below</p>
        </div>

        <!-- Feature Selection Tabs -->
        <div class="feature-tabs">
            <button class="feature-tab active" onclick="switchFeature('websearch')">
                <i class="fas fa-globe"></i> Web Search + Academic APIs
            </button>
            <button class="feature-tab" onclick="switchFeature('similarity')">
                <i class="fas fa-search"></i> Similarity Detection (all-mpnet-base-v2)
            </button>
        </div>

        <div class="feature-content">
            <!-- WEB SEARCH FEATURE -->
            <div id="websearch-panel" class="feature-panel active">
                <div class="web-search-header">
                    <h2><i class="fas fa-globe"></i> Web Search + Academic Database</h2>
                    <p style="color: #ccc;">Upload a PDF to check against web sources and academic papers</p>
                    <p style="color: #3182ce; font-size: 0.9em; margin-top: 5px;">Using Serper.dev + OpenAlex</p>
                </div>

                <div class="upload-area" id="webUploadArea" onclick="document.getElementById('webFileInput').click()">
                    <div class="upload-icon"><i class="fas fa-cloud-upload-alt"></i></div>
                    <div style="font-size: 1.2em; margin-bottom: 10px;">Drag & drop your PDF here or click to browse</div>
                    <div style="color: #ccc;">Supports PDF files only (max 10MB)</div>
                    <input type="file" id="webFileInput" accept=".pdf" style="display: none;">
                </div>

                <div id="webLoading" class="loading" style="display: none;">
                    <div class="spinner"></div>
                    <p>Analyzing your document with real-time web search...</p>
                </div>

                <div id="webResults" style="display: none;">
                    <div class="stats-grid">
                        <div class="stat-card">
                            <div class="stat-value" id="overallScore">0%</div>
                            <div>Overall Similarity</div>
                        </div>
                        <div class="stat-card">
                            <div class="stat-value" style="color: var(--web-blue);" id="webScore">0%</div>
                            <div>Web Sources</div>
                        </div>
                        <div class="stat-card">
                            <div class="stat-value" style="color: var(--academic-green);" id="academicScore">0%</div>
                            <div>Academic Papers</div>
                        </div>
                    </div>

                    <div class="legend">
                        <div class="legend-item">
                            <div class="color-box" style="background: rgba(49, 130, 206, 0.3); border: 2px solid var(--web-blue);"></div>
                            <span>Web Sources (click for URL)</span>
                        </div>
                        <div class="legend-item">
                            <div class="color-box" style="background: rgba(56, 161, 105, 0.3); border: 2px solid var(--academic-green);"></div>
                            <span>Academic Papers (hover for authors)</span>
                        </div>
                    </div>

                    <h3 style="color: var(--gold); margin: 20px 0 10px;">📝 Highlighted Text</h3>
                    <div class="highlight-box" id="highlightedText"></div>

                    <h3 style="color: var(--gold); margin: 20px 0 10px;">🔗 Matched Sources</h3>
                    <div id="sourcesList" class="sources-list"></div>

                    <button class="btn" onclick="downloadWebReport()" style="width: 100%;">
                        <i class="fas fa-download"></i> Download PDF Report
                    </button>
                </div>
            </div>

            <!-- SIMILARITY DETECTION FEATURE -->
            <div id="similarity-panel" class="feature-panel">
                <div class="similarity-header">
                    <h2><i class="fas fa-search"></i> Similarity Detection</h2>
                    <p style="color: #ccc;">Compare texts and documents using all-mpnet-base-v2 model</p>
                </div>

                <div class="mode-tabs">
                    <button class="mode-tab active" onclick="switchMode('text')">Text Input</button>
                    <button class="mode-tab" onclick="switchMode('one-to-one')">One-to-One Files</button>
                    <button class="mode-tab" onclick="switchMode('one-to-many')">One-to-Many Files</button>
                </div>

                <!-- Text Input Mode -->
                <div id="text-mode" class="tab-content active">
                    <div class="input-section">
                        <div class="text-input">
                            <h3>Original Text</h3>
                            <textarea id="text1" placeholder="Enter original text..."></textarea>
                        </div>
                        <div class="text-input">
                            <h3>Text to Check</h3>
                            <textarea id="text2" placeholder="Enter text to check..."></textarea>
                        </div>
                    </div>
                    <button class="btn" onclick="analyzeText()" style="width: 100%;">Analyze Text Similarity</button>
                </div>

                <!-- One-to-One File Mode -->
                <div id="one-to-one-mode" class="tab-content">
                    <div class="input-section">
                        <div class="text-input">
                            <h3>Document 1</h3>
                            <input type="file" id="file1" accept=".txt,.pdf,.docx">
                        </div>
                        <div class="text-input">
                            <h3>Document 2</h3>
                            <input type="file" id="file2" accept=".txt,.pdf,.docx">
                        </div>
                    </div>
                    <button class="btn" onclick="analyzeOneToOne()" style="width: 100%;">Compare Two Documents</button>
                </div>

                <!-- One-to-Many File Mode -->
                <div id="one-to-many-mode" class="tab-content">
                    <div class="input-section">
                        <div class="text-input">
                            <h3>Source Document</h3>
                            <input type="file" id="sourceFile" accept=".txt,.pdf,.docx">
                            <div class="file-list" id="sourceFileList"></div>
                        </div>
                        <div class="text-input">
                            <h3>Documents to Compare</h3>
                            <input type="file" id="compareFiles" multiple accept=".txt,.pdf,.docx">
                            <div class="file-list" id="compareFileList"></div>
                        </div>
                    </div>
                    <button class="btn" onclick="analyzeOneToMany()" style="width: 100%;">Compare Source vs All</button>
                </div>

                <!-- Progress Bar -->
                <div class="progress" style="margin-top: 30px;">
                    <div class="progress-bar" id="progressBar" style="width: 0%">0%</div>
                </div>

                <!-- Similarity Results -->
                <div id="similarityResults" style="margin-top: 30px; display: none;">
                    <div style="text-align: center; margin-bottom: 30px;">
                        <div style="width: 150px; height: 150px; border-radius: 50%; margin: 0 auto 15px; background: conic-gradient(from 0deg, #ff4444 0%, #ffaa00 50%, #00ff00 100%); display: flex; align-items: center; justify-content: center;">
                            <div style="width: 130px; height: 130px; background: var(--black); border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 2em; font-weight: bold; color: var(--gold); border: 3px solid var(--gold);">
                                <span id="similarityPercent">0%</span>
                            </div>
                        </div>
                        <h3 style="color: var(--gold);">Similarity Score</h3>
                    </div>
                    <div id="similarityHighlightContainer"></div>
                    <div id="oneToManyGrid" class="comparison-grid"></div>
                </div>
            </div>
        </div>
    </div>

    <script>
        let uploadedFiles = {
            source: null,
            compare: []
        };

        // Switch between main features
        function switchFeature(feature) {
            document.querySelectorAll('.feature-tab').forEach(t => t.classList.remove('active'));
            event.target.classList.add('active');

            document.querySelectorAll('.feature-panel').forEach(p => p.classList.remove('active'));
            document.getElementById(feature + '-panel').classList.add('active');
        }

        // Switch between modes in similarity feature
        function switchMode(mode) {
            document.querySelectorAll('.mode-tab').forEach(t => t.classList.remove('active'));
            event.target.classList.add('active');

            document.querySelectorAll('#similarity-panel .tab-content').forEach(c => c.classList.remove('active'));
            document.getElementById(mode + '-mode').classList.add('active');
            document.getElementById('similarityResults').style.display = 'none';
        }

        // ===== WEB SEARCH FEATURE FUNCTIONS =====
        document.getElementById('webFileInput').addEventListener('change', function(e) {
            if (e.target.files.length > 0) {
                uploadWebFile(e.target.files[0]);
            }
        });

        document.getElementById('webUploadArea').addEventListener('dragover', function(e) {
            e.preventDefault();
            this.style.borderColor = '#3182ce';
            this.style.background = 'rgba(49, 130, 206, 0.1)';
        });

        document.getElementById('webUploadArea').addEventListener('dragleave', function(e) {
            e.preventDefault();
            this.style.borderColor = '#3182ce';
            this.style.background = 'rgba(42, 42, 42, 0.8)';
        });

        document.getElementById('webUploadArea').addEventListener('drop', function(e) {
            e.preventDefault();
            this.style.borderColor = '#3182ce';
            this.style.background = 'rgba(42, 42, 42, 0.8)';

            const file = e.dataTransfer.files[0];
            if (file && file.type === 'application/pdf') {
                uploadWebFile(file);
            } else {
                alert('Please upload a PDF file');
            }
        });

        async function uploadWebFile(file) {
            const formData = new FormData();
            formData.append('file', file);

            document.getElementById('webUploadArea').style.display = 'none';
            document.getElementById('webLoading').style.display = 'block';
            document.getElementById('webResults').style.display = 'none';

            try {
                const response = await fetch('/analyze-web-pdf', {
                    method: 'POST',
                    body: formData
                });
                const data = await response.json();

                document.getElementById('webLoading').style.display = 'none';
                document.getElementById('webResults').style.display = 'block';

                document.getElementById('overallScore').textContent = data.overall_similarity + '%';
                document.getElementById('webScore').textContent = data.web_percentage + '%';
                document.getElementById('academicScore').textContent = data.academic_percentage + '%';
                document.getElementById('highlightedText').innerHTML = data.highlighted_text;

                const sourcesList = document.getElementById('sourcesList');
                sourcesList.innerHTML = '';
                data.sources.forEach(source => {
                    const item = document.createElement('div');
                    item.className = 'source-item';

                    let authorsHtml = '';
                    if (source.authors) {
                        authorsHtml = `<div style="margin-top: 5px; color: #aaa; font-size: 0.85em;">Authors: ${source.authors.join(', ').substring(0, 100)}</div>`;
                    }

                    item.innerHTML = `
                        <div class="source-title">${source.title}</div>
                        <a href="${source.url}" target="_blank" class="source-url">${source.url}</a>
                        <div style="margin-top: 5px; color: #ccc;">Confidence: ${Math.round(source.confidence * 100)}%</div>
                        ${authorsHtml}
                    `;
                    sourcesList.appendChild(item);
                });
            } catch (error) {
                alert('Error analyzing file');
                document.getElementById('webLoading').style.display = 'none';
                document.getElementById('webUploadArea').style.display = 'block';
            }
        }

        async function downloadWebReport() {
            window.location.href = '/download-web-report';
        }

        // ===== SIMILARITY DETECTION FUNCTIONS =====

        // File handling for one-to-many
        document.getElementById('sourceFile')?.addEventListener('change', function(e) {
            if (e.target.files.length > 0) {
                uploadedFiles.source = e.target.files[0];
                updateSourceFileList();
            }
        });

        document.getElementById('compareFiles')?.addEventListener('change', function(e) {
            uploadedFiles.compare = Array.from(e.target.files);
            updateCompareFileList();
        });

        function updateSourceFileList() {
            const list = document.getElementById('sourceFileList');
            if (uploadedFiles.source) {
                list.innerHTML = `
                    <div class="file-item">
                        <span>${uploadedFiles.source.name}</span>
                        <button class="remove-file" onclick="removeSourceFile()">×</button>
                    </div>
                `;
            } else {
                list.innerHTML = '<p>No source file selected</p>';
            }
        }

        function updateCompareFileList() {
            const list = document.getElementById('compareFileList');
            if (uploadedFiles.compare.length > 0) {
                list.innerHTML = uploadedFiles.compare.map((file, index) => `
                    <div class="file-item">
                        <span>${file.name}</span>
                        <button class="remove-file" onclick="removeCompareFile(${index})">×</button>
                    </div>
                `).join('');
            } else {
                list.innerHTML = '<p>No files to compare</p>';
            }
        }

        function removeSourceFile() {
            uploadedFiles.source = null;
            document.getElementById('sourceFile').value = '';
            updateSourceFileList();
        }

        function removeCompareFile(index) {
            uploadedFiles.compare.splice(index, 1);
            updateCompareFileList();
        }

        async function analyzeText() {
            const text1 = document.getElementById('text1').value;
            const text2 = document.getElementById('text2').value;

            if (!text1 || !text2) {
                alert('Please enter both texts');
                return;
            }

            updateProgress(50);

            const formData = new FormData();
            formData.append('text1', text1);
            formData.append('text2', text2);

            try {
                const response = await fetch('/analyze-similarity-text', { method: 'POST', body: formData });
                const data = await response.json();
                updateProgress(100);
                displaySimilarityResults(data);
            } catch (error) {
                alert('Error analyzing text');
                updateProgress(0);
            }
        }

        async function analyzeOneToOne() {
            const file1 = document.getElementById('file1').files[0];
            const file2 = document.getElementById('file2').files[0];

            if (!file1 || !file2) {
                alert('Please select two files');
                return;
            }

            updateProgress(50);

            const formData = new FormData();
            formData.append('file1', file1);
            formData.append('file2', file2);

            try {
                const response = await fetch('/analyze-similarity-files', { method: 'POST', body: formData });
                const data = await response.json();
                updateProgress(100);
                displaySimilarityResults(data);
            } catch (error) {
                alert('Error analyzing files');
                updateProgress(0);
            }
        }

        async function analyzeOneToMany() {
            if (!uploadedFiles.source || uploadedFiles.compare.length === 0) {
                alert('Please select source and comparison files');
                return;
            }

            updateProgress(50);

            const formData = new FormData();
            formData.append('source_file', uploadedFiles.source);
            uploadedFiles.compare.forEach(file => formData.append('compare_files', file));

            try {
                const response = await fetch('/analyze-similarity-many', { method: 'POST', body: formData });
                const data = await response.json();
                updateProgress(100);
                displayOneToManyResults(data.results);
            } catch (error) {
                alert('Error analyzing documents');
                updateProgress(0);
            }
        }

        function updateProgress(percent) {
            const progressBar = document.getElementById('progressBar');
            progressBar.style.width = percent + '%';
            progressBar.textContent = percent + '%';
        }

        function displaySimilarityResults(data) {
            document.getElementById('similarityPercent').textContent = data.percent.toFixed(1) + '%';

            const container = document.getElementById('similarityHighlightContainer');
            container.innerHTML = `
                <div class="text-comparison">
                    <div class="text-column">
                        <h6>Original Text (Highlighted):</h6>
                        <div class="highlighted-text">${data.highlighted_source}</div>
                    </div>
                    <div class="text-column">
                        <h6>Compared Text (Highlighted):</h6>
                        <div class="highlighted-text">${data.highlighted_compare}</div>
                    </div>
                </div>
            `;

            document.getElementById('similarityResults').style.display = 'block';
            document.getElementById('oneToManyGrid').style.display = 'none';
        }

        function displayOneToManyResults(results) {
            let html = '<div class="comparison-grid">';

            results.forEach((result, index) => {
                const badgeClass = result.similarity >= 70 ? 'badge-high' :
                                 result.similarity >= 30 ? 'badge-medium' : 'badge-low';

                html += `
                    <div class="comparison-card">
                        <div class="similarity-badge ${badgeClass}">
                            ${result.similarity.toFixed(1)}% Similarity
                        </div>
                        <div style="margin-bottom: 10px;">
                            <strong>With:</strong> ${result.compared_with}
                        </div>
                        <button class="expand-btn" onclick="toggleComparison(${index})">
                            Show Details
                        </button>
                        <div id="comparison-${index}" class="hidden">
                            <div class="text-comparison" style="margin-top: 15px;">
                                <div class="text-column">
                                    <h6>Source:</h6>
                                    <div class="highlighted-text">${result.highlighted_source}</div>
                                </div>
                                <div class="text-column">
                                    <h6>Compared:</h6>
                                    <div class="highlighted-text">${result.highlighted_compare}</div>
                                </div>
                            </div>
                        </div>
                    </div>
                `;
            });

            html += '</div>';

            document.getElementById('oneToManyGrid').innerHTML = html;
            document.getElementById('similarityResults').style.display = 'block';
            document.getElementById('oneToManyGrid').style.display = 'block';
            document.getElementById('similarityHighlightContainer').style.display = 'none';
        }

        function toggleComparison(index) {
            const div = document.getElementById(`comparison-${index}`);
            const btn = event.target;
            if (div.classList.contains('hidden')) {
                div.classList.remove('hidden');
                btn.textContent = 'Hide Details';
            } else {
                div.classList.add('hidden');
                btn.textContent = 'Show Details';
            }
        }
    </script>
</body>
</html>
'''

@app.route('/')
def home():
    """Home page route"""
    return render_template_string(HTML_TEMPLATE)

# ===== WEB SEARCH ROUTES =====

@app.route('/analyze-web-pdf', methods=['POST'])
def analyze_web_pdf():
    """Improved web search PDF analysis with confidence normalization"""
    global last_web_results, last_web_sources, last_filename, last_upload_time

    try:
        if 'file' not in request.files:
            return jsonify({'error': 'No file uploaded'}), 400

        file = request.files['file']
        last_filename = file.filename
        last_upload_time = datetime.now()

        if not file.filename.endswith('.pdf'):
            return jsonify({'error': 'Please upload a PDF file'}), 400

        print(f"📄 Processing file: {file.filename}")

        # Use your existing web search analyzer
        results = analyzer.analyze_document(file)

        if results is None:
            return jsonify({'error': 'Could not process document - no text extracted'}), 400

        # Store results for report generation
        last_web_results = results

        # Generate highlighted HTML
        highlighted_html = analyzer.generate_highlighted_html(results)

        # Prepare sources with confidence normalization
        sources = []
        seen_urls = set()

        for match in results['all_matched_sources']:
            url = match.get('url', match.get('doi', ''))
            if url and url not in seen_urls:
                seen_urls.add(url)

                # Format title
                title = match.get('title', 'Unknown Source')
                if len(title) > 100:
                    title = title[:100] + '...'

                # Normalize confidence (ensure between 0 and 1)
                confidence = match.get('confidence', 0)
                if confidence is None:
                    confidence = 0
                if confidence > 1:
                    confidence = confidence / 100.0
                confidence = max(0.0, min(1.0, confidence))

                source_info = {
                    'title': title,
                    'url': url,
                    'source_type': match['source_type'],
                    'confidence': round(confidence, 2)
                }

                # Add authors for academic sources
                if match['source_type'] == 'academic' and match.get('authors'):
                    source_info['authors'] = match['authors']

                sources.append(source_info)

        # Sort sources by confidence
        sources.sort(key=lambda x: x['confidence'], reverse=True)
        last_web_sources = sources

        response_data = {
            'overall_similarity': results['overall_similarity'],
            'web_percentage': results['web_percentage'],
            'academic_percentage': results['academic_percentage'],
            'highlighted_text': highlighted_html,
            'sources': sources[:20]  # Limit to top 20 sources
        }

        print(f"✅ Analysis complete: {len(sources)} sources found")
        return jsonify(response_data)

    except Exception as e:
        print(f"❌ Error in analyze-web-pdf: {e}")
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@app.route('/download-web-report', methods=['GET'])
def download_web_report():
    """Enhanced PDF report with highlighted text and source URLs (confidence normalized)"""
    try:
        from reportlab.lib import colors
        from reportlab.lib.pagesizes import letter, A4
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.units import inch
        from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
        import textwrap
        from datetime import datetime

        if last_web_results is None or last_web_sources is None:
            # Create a simple error report
            buffer = io.BytesIO()
            doc = SimpleDocTemplate(buffer, pagesize=A4)
            styles = getSampleStyleSheet()
            story = []
            story.append(Paragraph("Academic Integrity Web Search Report", styles['Title']))
            story.append(Spacer(1, 12))
            story.append(Paragraph("No analysis data available. Please analyze a document first.", styles['Normal']))
            doc.build(story)
            buffer.seek(0)
            return send_file(
                buffer,
                as_attachment=True,
                download_name='error_report.pdf',
                mimetype='application/pdf'
            )

        # Create PDF buffer
        buffer = io.BytesIO()

        # Create document
        doc = SimpleDocTemplate(
            buffer,
            pagesize=A4,
            rightMargin=72,
            leftMargin=72,
            topMargin=72,
            bottomMargin=72,
        )

        # Styles
        styles = getSampleStyleSheet()

        # Custom styles
        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=24,
            textColor=colors.HexColor('#FFD700'),
            alignment=TA_CENTER,
            spaceAfter=30,
            fontName='Helvetica-Bold'
        )

        heading_style = ParagraphStyle(
            'CustomHeading',
            parent=styles['Heading2'],
            fontSize=16,
            textColor=colors.HexColor('#FFD700'),
            spaceAfter=12,
            spaceBefore=20,
            fontName='Helvetica-Bold'
        )

        subheading_style = ParagraphStyle(
            'Subheading',
            parent=styles['Heading3'],
            fontSize=14,
            textColor=colors.HexColor('#CCCCCC'),
            spaceAfter=10,
            spaceBefore=15,
            fontName='Helvetica-Bold'
        )

        normal_style = ParagraphStyle(
            'CustomNormal',
            parent=styles['Normal'],
            fontSize=10,
            textColor=colors.HexColor('#FFFFFF'),
            spaceAfter=8,
            fontName='Helvetica'
        )

        # Style for web source text (blue)
        web_text_style = ParagraphStyle(
            'WebText',
            parent=styles['Normal'],
            fontSize=10,
            textColor=colors.HexColor('#3182ce'),
            backColor=colors.Color(0.19, 0.51, 0.81, 0.1),
            spaceAfter=8,
            leftIndent=10,
            fontName='Helvetica'
        )

        # Style for academic source text (green)
        academic_text_style = ParagraphStyle(
            'AcademicText',
            parent=styles['Normal'],
            fontSize=10,
            textColor=colors.HexColor('#38a169'),
            backColor=colors.Color(0.22, 0.63, 0.41, 0.1),
            spaceAfter=8,
            leftIndent=10,
            fontName='Helvetica'
        )

        source_title_style = ParagraphStyle(
            'SourceTitle',
            parent=styles['Normal'],
            fontSize=11,
            textColor=colors.HexColor('#FFD700'),
            spaceAfter=4,
            fontName='Helvetica-Bold'
        )

        source_url_style = ParagraphStyle(
            'SourceURL',
            parent=styles['Normal'],
            fontSize=9,
            textColor=colors.HexColor('#3182ce'),
            spaceAfter=2,
            fontName='Helvetica'
        )

        # Build story
        story = []

        # Title
        story.append(Paragraph("🔍 Academic Integrity Web Search Report", title_style))
        story.append(Spacer(1, 0.2*inch))

        # Report Metadata
        data = [
            ['Report Generated:', datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            ['Document Analyzed:', last_filename if last_filename else 'Unknown'],
            ['Analysis Method:', 'Serper.dev (Web) + OpenAlex (Academic)'],
        ]

        table = Table(data, colWidths=[2*inch, 4*inch])
        table.setStyle(TableStyle([
            ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 0), (-1, -1), 10),
            ('TEXTCOLOR', (0, 0), (0, -1), colors.HexColor('#FFD700')),
            ('TEXTCOLOR', (1, 0), (1, -1), colors.HexColor('#FFFFFF')),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ]))
        story.append(table)
        story.append(Spacer(1, 0.3*inch))

        # Similarity Scores
        story.append(Paragraph("📊 Similarity Analysis", heading_style))

        scores_data = [
            ['Source Type', 'Percentage', 'Status'],
            ['Overall Similarity', f"{last_web_results['overall_similarity']}%", ''],
            ['Web Sources', f"{last_web_results['web_percentage']}%", ''],
            ['Academic Papers', f"{last_web_results['academic_percentage']}%", ''],
            ['Original Content', f"{100 - last_web_results['overall_similarity']}%", '']
        ]

        scores_table = Table(scores_data, colWidths=[2.5*inch, 1.5*inch, 2*inch])
        scores_table.setStyle(TableStyle([
            ('FONTNAME', (0, 0), (-1, -1), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 11),
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#FFD700')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.HexColor('#000000')),
            ('TEXTCOLOR', (1, 1), (1, -1), colors.HexColor('#FFFFFF')),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
            ('GRID', (0, 0), (-1, -1), 1, colors.HexColor('#333333')),
        ]))

        # Color code the percentages
        scores_table.setStyle(TableStyle([
            ('TEXTCOLOR', (1, 2), (1, 2), colors.HexColor('#3182ce')),
            ('TEXTCOLOR', (1, 3), (1, 3), colors.HexColor('#38a169')),
        ]))

        story.append(scores_table)
        story.append(Spacer(1, 0.3*inch))

        # Color Legend
        story.append(Paragraph("📋 Color Legend", subheading_style))
        legend_data = [
            ['🔵 Blue:', 'Web Sources - Content found on websites'],
            ['🟢 Green:', 'Academic Papers - Content from scholarly articles'],
        ]
        legend_table = Table(legend_data, colWidths=[1.5*inch, 5*inch])
        legend_table.setStyle(TableStyle([
            ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 0), (-1, -1), 10),
            ('TEXTCOLOR', (0, 0), (0, -1), colors.HexColor('#3182ce')),
            ('TEXTCOLOR', (1, 1), (1, 1), colors.HexColor('#38a169')),
            ('TEXTCOLOR', (1, 0), (1, 0), colors.HexColor('#3182ce')),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
        ]))
        story.append(legend_table)
        story.append(Spacer(1, 0.3*inch))

        # Highlighted Text Section
        story.append(Paragraph("📝 Highlighted Text with Sources", heading_style))
        story.append(Paragraph("The following text shows matches found in your document:", normal_style))
        story.append(Spacer(1, 0.1*inch))

        # Add highlighted text with source markers (confidence already normalized)
        for chunk in last_web_results['chunk_results']:
            if chunk['source_type'] == 'web':
                if chunk.get('best_match'):
                    best = chunk['best_match']
                    story.append(Paragraph(f"🔵 [WEB] {chunk['text']}", web_text_style))
                    story.append(Paragraph(f"    📎 Source: {best.get('url', 'No URL')}", source_url_style))
                else:
                    story.append(Paragraph(f"🔵 [WEB] {chunk['text']}", web_text_style))
            elif chunk['source_type'] == 'academic':
                if chunk.get('best_match'):
                    best = chunk['best_match']
                    authors = ', '.join(best.get('authors', ['Unknown'])[:2])
                    story.append(Paragraph(f"🟢 [ACADEMIC] {chunk['text']}", academic_text_style))
                    story.append(Paragraph(f"    📚 Authors: {authors}", source_url_style))
                    if best.get('doi'):
                        story.append(Paragraph(f"    🔗 DOI: https://doi.org/{best['doi']}", source_url_style))
                else:
                    story.append(Paragraph(f"🟢 [ACADEMIC] {chunk['text']}", academic_text_style))
            else:
                story.append(Paragraph(chunk['text'], normal_style))
            story.append(Spacer(1, 0.1*inch))

        story.append(PageBreak())

        # Sources Section
        story.append(Paragraph("🔗 Detailed Source List", heading_style))
        story.append(Paragraph(f"Found {len(last_web_sources)} matching sources:", normal_style))
        story.append(Spacer(1, 0.2*inch))

        if last_web_sources:
            for i, source in enumerate(last_web_sources[:20], 1):
                # Normalize confidence again to be safe
                conf = source.get('confidence', 0)
                if conf is None:
                    conf = 0
                if conf > 1:
                    conf = conf / 100.0
                conf = max(0.0, min(1.0, conf))
                conf_percent = int(conf * 100)

                if source['source_type'] == 'web':
                    story.append(Paragraph(f"{i}. 🌐 {source['title']}", source_title_style))
                    story.append(Paragraph(f"   URL: {source['url']}", source_url_style))
                else:
                    story.append(Paragraph(f"{i}. 📚 {source['title']}", source_title_style))
                    story.append(Paragraph(f"   URL: {source['url']}", source_url_style))
                    if source.get('authors'):
                        authors_str = ', '.join(source['authors'][:3])
                        story.append(Paragraph(f"   Authors: {authors_str}", normal_style))

                # Confidence bar
                bar_length = int(conf * 30)
                bar = '█' * bar_length + '░' * (30 - bar_length)
                story.append(Paragraph(f"   Confidence: {bar} {conf_percent}%", normal_style))
                story.append(Spacer(1, 0.1*inch))
        else:
            story.append(Paragraph("No sources found.", normal_style))

        story.append(Spacer(1, 0.3*inch))

        # Footer with summary
        story.append(Paragraph("—" * 70, normal_style))

        web_matches = len([s for s in last_web_sources if s['source_type'] == 'web'])
        academic_matches = len([s for s in last_web_sources if s['source_type'] == 'academic'])

        summary_text = f"""
        📊 REPORT SUMMARY
        • Total Similarity: {last_web_results['overall_similarity']}%
        • Web Sources: {last_web_results['web_percentage']}% ({web_matches} matches)
        • Academic Papers: {last_web_results['academic_percentage']}% ({academic_matches} matches)
        • Total Sources Found: {len(last_web_sources)}

        Generated by Academic Integrity System using Serper.dev + OpenAlex
        """
        story.append(Paragraph(summary_text, normal_style))

        # Build PDF
        doc.build(story)
        buffer.seek(0)

        # Generate filename with timestamp
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'academic_integrity_report_{timestamp}.pdf'

        return send_file(
            buffer,
            as_attachment=True,
            download_name=filename,
            mimetype='application/pdf'
        )

    except Exception as e:
        print(f"Error generating PDF: {e}")
        import traceback
        traceback.print_exc()

        # Fallback simple report
        buffer = io.BytesIO()
        doc = SimpleDocTemplate(buffer, pagesize=letter)
        styles = getSampleStyleSheet()
        story = []
        story.append(Paragraph("Academic Integrity Report", styles['Title']))
        story.append(Spacer(1, 12))
        if last_web_results:
            story.append(Paragraph(f"Overall Similarity: {last_web_results['overall_similarity']}%", styles['Normal']))
            story.append(Paragraph(f"Web Sources: {last_web_results['web_percentage']}%", styles['Normal']))
            story.append(Paragraph(f"Academic Papers: {last_web_results['academic_percentage']}%", styles['Normal']))
        else:
            story.append(Paragraph("No data available", styles['Normal']))
        doc.build(story)
        buffer.seek(0)
        return send_file(buffer, as_attachment=True, download_name='report.pdf', mimetype='application/pdf')

# ===== SIMILARITY DETECTION ROUTES =====

@app.route('/analyze-similarity-text', methods=['POST'])
def analyze_similarity_text():
    """Text-to-text similarity comparison"""
    try:
        text1 = request.form.get("text1", "")
        text2 = request.form.get("text2", "")

        if not text1 or not text2:
            return jsonify({"error": "Please provide both texts"}), 400

        emb1 = model.encode(text1, convert_to_tensor=True)
        emb2 = model.encode(text2, convert_to_tensor=True)
        score = util.cos_sim(emb1, emb2).item()
        percent = score * 100

        highlighted_source, highlighted_compare, _ = highlight_similar_text(text1, text2)

        return jsonify({
            "score": score,
            "percent": percent,
            "highlighted_source": highlighted_source,
            "highlighted_compare": highlighted_compare
        })

    except Exception as e:
        print(f"Error in analyze-similarity-text: {e}")
        return jsonify({"error": str(e)}), 500

@app.route('/analyze-similarity-files', methods=['POST'])
def analyze_similarity_files():
    """One-to-one file comparison"""
    try:
        file1 = request.files.get("file1")
        file2 = request.files.get("file2")

        if not file1 or not file2:
            return jsonify({"error": "Please upload both files"}), 400

        text1 = read_file(file1)
        text2 = read_file(file2)

        if not text1 or not text2:
            return jsonify({"error": "Could not read file contents"}), 400

        emb1 = model.encode(text1, convert_to_tensor=True)
        emb2 = model.encode(text2, convert_to_tensor=True)
        score = util.cos_sim(emb1, emb2).item()
        percent = score * 100

        highlighted_source, highlighted_compare, _ = highlight_similar_text(text1, text2)

        return jsonify({
            "score": score,
            "percent": percent,
            "file1": file1.filename,
            "file2": file2.filename,
            "highlighted_source": highlighted_source,
            "highlighted_compare": highlighted_compare
        })

    except Exception as e:
        print(f"Error in analyze-similarity-files: {e}")
        return jsonify({"error": str(e)}), 500

@app.route('/analyze-similarity-many', methods=['POST'])
def analyze_similarity_many():
    """One-to-many file comparison"""
    try:
        source_file = request.files.get("source_file")
        compare_files = request.files.getlist("compare_files")

        if not source_file or len(compare_files) == 0:
            return jsonify({"error": "Please upload source and comparison files"}), 400

        source_content = read_file(source_file)
        if not source_content:
            return jsonify({"error": "Could not read source file"}), 400

        results = []
        for file in compare_files:
            if file and file.filename:
                content = read_file(file)
                if content:
                    source_emb = model.encode(source_content, convert_to_tensor=True)
                    compare_emb = model.encode(content, convert_to_tensor=True)
                    score = util.cos_sim(source_emb, compare_emb).item()
                    similarity = score * 100

                    highlighted_source, highlighted_compare, _ = highlight_similar_text(source_content, content)

                    results.append({
                        'source': source_file.filename,
                        'compared_with': file.filename,
                        'similarity': similarity,
                        'highlighted_source': highlighted_source,
                        'highlighted_compare': highlighted_compare
                    })

        return jsonify({"results": results})

    except Exception as e:
        print(f"Error in analyze-similarity-many: {e}")
        return jsonify({"error": str(e)}), 500

print("✅ Flask app created with two separate features and confidence normalization!")
print("   Routes configured:")
print("   - / (Home page)")
print("   - /analyze-web-pdf (Web search PDF analysis)")
print("   - /download-web-report (Download PDF report with highlights)")
print("   - /analyze-similarity-text (Text similarity)")
print("   - /analyze-similarity-files (One-to-one file comparison)")
print("   - /analyze-similarity-many (One-to-many file comparison)")

✅ Flask app created with two separate features and confidence normalization!
   Routes configured:
   - / (Home page)
   - /analyze-web-pdf (Web search PDF analysis)
   - /download-web-report (Download PDF report with highlights)
   - /analyze-similarity-text (Text similarity)
   - /analyze-similarity-files (One-to-one file comparison)
   - /analyze-similarity-many (One-to-many file comparison)


In [ ]:
# @title Cell 8: Launch the Application

# Open ngrok tunnel
public_url = ngrok.connect(5000)
print(f"🌍 Your application is live at: {public_url}")
print("\n📱 Open this URL in your browser")
print("🔑 Using your Serper API key: 8eff6c07...")
print("\n✨ Two Features Available:")
print("   1. 🌐 WEB SEARCH + ACADEMIC APIS (Your Original Feature)")
print("      - Upload PDF, get real-time web search results")
print("      - Blue highlighting for web sources, Green for academic")
print("      - Download PDF report with sources")
print("\n   2. 🔍 SIMILARITY DETECTION (New Feature)")
print("      - Text-to-Text comparison")
print("      - One-to-One file comparison")
print("      - One-to-Many file comparison")
print("      - Uses all-mpnet-base-v2 model")
print("      - Red highlighting for similar content")

# Run Flask app
app.run(port=5000, use_reloader=False)

🌍 Your application is live at: NgrokTunnel: "https://549a-34-83-54-229.ngrok-free.app" -> "http://localhost:5000"

📱 Open this URL in your browser
🔑 Using your Serper API key: 8eff6c07...

✨ Two Features Available:
   1. 🌐 WEB SEARCH + ACADEMIC APIS (Your Original Feature)
      - Upload PDF, get real-time web search results
      - Blue highlighting for web sources, Green for academic
      - Download PDF report with sources

   2. 🔍 SIMILARITY DETECTION (New Feature)
      - Text-to-Text comparison
      - One-to-One file comparison
      - One-to-Many file comparison
      - Uses all-mpnet-base-v2 model
      - Red highlighting for similar content
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [09/Apr/2026 16:47:27] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Apr/2026 16:47:29] "GET /favicon.ico HTTP/1.1" 404 -


📄 Processing file: sample.pdf
✅ Extracted 1 text blocks from PDF
✅ Created 14 text chunks for analysis
📄 Analyzing 14 chunks...
   Processing chunk 1/14...
OpenAlex API error: HTTPSConnectionPool(host='api.openalex.org', port=443): Read timed out. (read timeout=15)
   Processing chunk 6/14...
   Processing chunk 11/14...


INFO:werkzeug:127.0.0.1 - - [09/Apr/2026 16:48:27] "POST /analyze-web-pdf HTTP/1.1" 200 -


✅ Analysis complete: 2 web matches, 6 academic matches
✅ Analysis complete: 9 sources found


INFO:werkzeug:127.0.0.1 - - [09/Apr/2026 16:50:46] "GET / HTTP/1.1" 200 -


📄 Processing file: GROUP-10 (AcademicIntegrityPoster).pptx.pdf
✅ Extracted 1 text blocks from PDF
✅ Created 14 text chunks for analysis
📄 Analyzing 14 chunks...
   Processing chunk 1/14...
OpenAlex API error: HTTPSConnectionPool(host='api.openalex.org', port=443): Read timed out. (read timeout=15)
   Processing chunk 6/14...
   Processing chunk 11/14...


INFO:werkzeug:127.0.0.1 - - [09/Apr/2026 16:51:51] "POST /analyze-web-pdf HTTP/1.1" 200 -


OpenAlex API error: 400 Client Error: Bad Request for url: https://api.openalex.org/works?search=in+accuracy+significantly+beyond+keyword+matching+academic+data+sources+educational+institutions+UNDER+THE+GUIDANCE+OF%3A+Dr.Dhawaleswar+Rao+%7C+Centurion+University+of+Technology+and+Management+%7C+GROUP+%3A&per-page=3&sort=relevance_score%3Adesc&mailto=ashishnayakofficial05%40gmail.com
✅ Analysis complete: 0 web matches, 0 academic matches
✅ Analysis complete: 13 sources found
